# Pix2Struct TextCaps-base — DIMER text-aware image captioning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/pix2struct-textcaps-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/pix2struct-textcaps-pipeline/blob/main/tutorials/pix2struct_textcaps_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Fpix2struct--textcaps--base-ffcc4d?style=flat)](https://huggingface.co/google/pix2struct-textcaps-base) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Fpix2struct-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/pix2struct) [![arXiv](https://img.shields.io/badge/arXiv-2210.03347-b31b1b.svg)](https://arxiv.org/abs/2210.03347)

**Profile:** `TASK-INFERENCE`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** Text-aware image captioning — one image (optionally with a caption prefix to continue) → one sentence that quotes the text visible in the image — using the pinned `google/pix2struct-textcaps-base` weights

**This notebook is standalone.** It carries the repository's pipeline module (`src/pix2struct_textcaps_pipeline/pipeline.py` at revision `d2f9b7c09197`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `61bee0d7e2378e601b68f853ceee4f7cf99f1b88` (~1133 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned snapshot, obtains the tutorial sample automatically, validates it into an input manifest before the model runs, runs the task locally in this kernel, writes the evaluation report, and exports machine-readable outputs with provenance. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5).

**Bring Your Own Data:** After the sample workflow completes, set `USE_BYOD = True` in the sample cell and re-run from that cell to supply your own input. It passes through the same notebook-local validation, task, evaluation-report and export cells as the sample; the expected input format, the ceilings and the privacy guidance are stated in the Prerequisites and in the sample cell, and the upload stays inside this runtime. BYOD is optional and never part of the default path.

At inference the Pix2Struct image-encoder/text-decoder (a ViT-style encoder over variable-resolution 16×16 patches and a 12-layer text decoder, 282M parameters, pretrained by parsing masked web screenshots into HTML and fine-tuned on TextCaps, whose captions must mention the text in the image) scales the image to fill at most 2048 patches, reads any text in it from pixels, and generates the caption token by token, either from scratch (unconditional) or continuing a prefix you supply such as `A picture of` (conditional, the upstream README's example). Decoding is greedy (`do_sample=False`, one beam) under a caller-owned `max_new_tokens` budget. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights, processor and tokenizer, and the carried module adds snapshot verification, the input contract (image side ceilings, an optional prefix up to 128 characters, the token budget), a fixed output contract, and the `text_recall`, `keyword_hits`, `unigram_f1`, `validate_inputs` and `evaluation_report` helpers. The default sample is three flat cartoon scenes drawn in code with **known text rendered into them**, so `text_recall` — the fraction of the drawn words the caption quotes — is demonstration (plumbing) evidence for the OCR-free reading path, not a TextCaps benchmark.

**Learning objectives:** install the pinned runtime, read what the carried pipeline module guarantees, resolve and digest-verify the immutable upstream model revision, draw three synthetic scenes with known text (or upload your own photographs of signs and labels) and validate them into an input manifest, choose a token budget and an optional prefix, run the supported task, read the captions correctly (generated text, no score, a `truncated` flag), exercise an optional BYOD path, produce an evaluation report that is `sample-sanity` with `text_recall` when the drawn text is known and `not-measurable` otherwise, and export the captions, an annotated contact sheet and provenance.

**This notebook does not demonstrate:** General OCR (the model quotes text it deems caption-worthy, not every string in the image, and returns no transcript or location), visual question answering and document QA (separate checkpoints), captions in languages other than English, batch throughput, sampling, beam search or repetition penalties (greedy decoding for reproducibility), evaluation on the TextCaps benchmark (not bundled; CIDEr and BLEU-4 need several references per image and are not computed here), and any training. The model was fine-tuned on photographs of signs, products, screens and labels; flat drawings, documents and diagrams are outside what this notebook measures, and a fluent wrong caption carries no signal.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. CPU is adequate: the repository's model card records 5.6 s to load and 2.8–3.4 s per caption on the drawn scenes in the Windows venv (Intel Core Ultra 9 275HX) — the 2048-patch encoder pass dominates. The pinned `torch==2.14.0` install and the 1.13 GB checkpoint are the large downloads of the run.
- **Knowledge:** basic Python and PIL; what an encoder–decoder model's generated tokens are; what reference-based caption metrics (CIDEr, BLEU) need and why token recall of drawn text is not one of them; that a confident caption is not a correct one.
- **Data:** the default sample is three deterministic cartoon scenes drawn in code with Pillow's bundled font (a red octagonal sign reading `STOP`; a storefront reading `Blue Fern Bakery` with an `OPEN` card; a green jersey reading `LIONS` and `42`), each with the list of strings drawn into it, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one or more images decodable by Pillow (PNG/JPEG/WebP and similar), any colour mode, sides between 16 and 4096 px; the drawn-text list is unknown for uploads, so their report is `not-measurable`. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/pix2struct-textcaps-base` snapshot (~1133 MB in total) at revision `61bee0d7e237…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'pix2struct-textcaps-pipeline',
    'repository_revision': 'd2f9b7c091976d355d0c30f41a3f83d87808e7d9',
    'embedded_module': 'src/pix2struct_textcaps_pipeline/pipeline.py',
    'embedded_modules': ['src/pix2struct_textcaps_pipeline/pipeline.py'],
    'module_sha256': '03336868dc1801cc5e98559dd0d9119ab865c3f5b844ed8c9ca66c32caa58bd8',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/pix2struct_textcaps_pipeline/` @ `d2f9b7c09197`)

The next 1 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/1:** `src/pix2struct_textcaps_pipeline/pipeline.py`

In [ ]:
"""Text-aware image captioning with the pinned ``google/pix2struct-textcaps-base`` checkpoint.

The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)
or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with
``trust_remote_code=False``: the Pix2Struct architecture comes from the pinned ``transformers`` release,
the weights are SafeTensors, and no model-repository code is executed. The TextCaps checkpoint reads the
text in the image from pixels and is expected to mention it in the caption; this package measures that
as ``text_recall`` on text the caller drew, which is sanity evidence, not a captioning metric.
"""

from __future__ import annotations

import hashlib
import json
import re
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "google/pix2struct-textcaps-base"
MODEL_REVISION = "61bee0d7e2378e601b68f853ceee4f7cf99f1b88"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "pix2struct-textcaps-base"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Generation ceilings. TextCaps captions are one sentence that quotes text in the image (the
# checkpoint's text_config max_length is 20); the default leaves room for a long quoted string and
# the ceiling bounds runaway generation.
MAX_NEW_TOKENS = 64
DEFAULT_MAX_NEW_TOKENS = 30
DECODING = "greedy"
# Optional conditional-captioning prefix (the upstream README's "A picture of"); the model continues
# it. A prefix longer than a short phrase is not what the model was trained on.
MAX_PREFIX_CHARS = 128
# Input ceilings. The processor extracts at most MAX_PATCHES 16x16 patches (preprocessor_config.json)
# after scaling the image to fill that budget (aspect ratio preserved), so pixel count only guards
# memory during resizing.
MAX_PATCHES = 2048
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 16
_PUNCT_RE = re.compile(r"[^\w\s]")


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def normalize_caption(text: str) -> str:
    """COCO-caption-style normalisation: lower-case, punctuation removed, whitespace collapsed."""
    return " ".join(_PUNCT_RE.sub(" ", text.lower()).split())


def caption_tokens(text: str) -> list[str]:
    return normalize_caption(text).split()


def unigram_f1(prediction: str, references: Sequence[str]) -> float:
    """Bag-of-words F1 between the normalised prediction and the best-matching reference.

    A plumbing check, not a captioning metric: CIDEr, BLEU-4 and SPICE need several references per
    image and corpus-level statistics. Multiset overlap counts repeated words once per occurrence.
    """
    if not references:
        raise ValueError("references must contain at least one caption")
    pred = caption_tokens(prediction)
    best = 0.0
    for reference in references:
        ref = caption_tokens(reference)
        if not pred or not ref:
            continue
        ref_counts: dict[str, int] = {}
        for token in ref:
            ref_counts[token] = ref_counts.get(token, 0) + 1
        overlap = 0
        for token in pred:
            if ref_counts.get(token, 0) > 0:
                overlap += 1
                ref_counts[token] -= 1
        if overlap:
            precision, recall = overlap / len(pred), overlap / len(ref)
            best = max(best, 2 * precision * recall / (precision + recall))
    return best


def keyword_hits(caption: str, keywords: Sequence[str]) -> dict[str, bool]:
    """Which of the caller's keywords (normalised, whole-token match) appear in the caption."""
    tokens = set(caption_tokens(caption))
    return {keyword: all(part in tokens for part in caption_tokens(keyword)) for keyword in keywords}


def text_recall(caption: str, drawn_texts: Sequence[str]) -> dict[str, Any]:
    """Fraction of the distinct normalised tokens of the drawn text strings that appear in the caption.

    The TextCaps task is to mention the text visible in the image; on text the caller drew, recall of
    those tokens is a plumbing check of the OCR-free reading path, not a captioning metric.
    """
    if not drawn_texts:
        raise ValueError("drawn_texts must contain at least one string")
    expected: list[str] = []
    for text in drawn_texts:
        for token in caption_tokens(text):
            if token not in expected:
                expected.append(token)
    if not expected:
        raise ValueError("drawn_texts contain no word tokens after normalisation")
    present = set(caption_tokens(caption))
    found = [token for token in expected if token in present]
    return {
        "recall": len(found) / len(expected),
        "found": found,
        "missing": [token for token in expected if token not in present],
        "n_expected": len(expected),
    }


def validate_image(image: Any) -> Image.Image:
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {min(width, height)} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {max(width, height)} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": "one image as PIL.Image.Image (any mode, converted to RGB) plus an optional caption prefix",
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "prefix_chars": [0, MAX_PREFIX_CHARS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "decoding": f"{DECODING} (do_sample=False, num_beams=1), deterministic on a fixed device and dtype",
    "preprocessing": (
        "image scaled to fill at most MAX_PATCHES 16x16 patches (aspect ratio preserved), normalised per "
        "image and flattened into patch tokens with row/column positions; the text decoder starts from the "
        "start token (unconditional) or from the tokenised prefix (conditional) and generates the caption"
    ),
    "output": "one caption string (the model's decoded text, prefix included when given), no score",
}


def _check_inputs(image: Any, prefix: Any, max_new_tokens: Any) -> tuple[Image.Image, str | None, int]:
    """Raise TypeError/ValueError naming the first violated ceiling; return the checked request.

    ``caption`` and ``validate_inputs`` both route through this function so their acceptance
    criteria cannot diverge.
    """
    rgb = validate_image(image)
    checked_prefix: str | None = None
    if prefix is not None:
        if not isinstance(prefix, str):
            raise TypeError("prefix must be a str or None")
        checked_prefix = " ".join(prefix.split())
        if not checked_prefix:
            raise ValueError("prefix must contain at least one non-whitespace character or be None")
        if len(checked_prefix) > MAX_PREFIX_CHARS:
            raise ValueError(f"prefix has {len(checked_prefix)} chars > MAX_PREFIX_CHARS {MAX_PREFIX_CHARS}")
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    return rgb, checked_prefix, max_new_tokens


def validate_inputs(
    images: Sequence[Image.Image],
    *,
    prefix: str | None = None,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Every image is checked exactly as ``caption`` would check it; rejection is reported by raising,
    and a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    if isinstance(images, Image.Image) or not isinstance(images, Sequence) or not images:
        raise TypeError("images must be a non-empty sequence of PIL.Image.Image")
    if names is not None and len(names) != len(images):
        raise ValueError(f"names has {len(names)} entries for {len(images)} images")
    checked_prefix = None
    observed = []
    for index, image in enumerate(images):
        _, checked_prefix, _ = _check_inputs(image, prefix, max_new_tokens)
        observed.append(
            {"id": names[index] if names else f"image-{index}", "mode": image.mode, "size": list(image.size)}
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": observed,
        "prefix": checked_prefix,
        "generation": {"max_new_tokens": int(max_new_tokens), "do_sample": False, "decoding": DECODING},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Sequence[Mapping[str, Any]],
    drawn_texts: Sequence[Sequence[str]] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``drawn_texts`` (one sequence of text strings known to be in each image, in order) the report
    carries the mean ``text_recall`` over the images plus one per-image entry, verdict
    ``sample-sanity``; without them it is ``not-measurable`` and says what labelled data would make the
    task measurable. Neither is a captioning benchmark.
    """
    if not results:
        raise ValueError("results must contain at least one caption result")
    base = {
        "task": "image (+ optional prefix) -> caption text that quotes the text in the image (TextCaps)",
        "score_semantics": (
            "the caption is generated text and carries no score, probability or correctness signal; a "
            "fluent caption is not evidence that it describes the image. Greedy decoding makes the output "
            "reproducible on a fixed device and dtype, a reproducibility property, not a quality one"
        ),
        "sample_kind": sample_kind,
        "n_images": len(results),
        "truncated": [bool(result.get("truncated")) for result in results],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if drawn_texts is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no drawn text or reference captions were supplied for the captioned images",
            "needs": (
                "several human-written reference captions per image from the deployment domain "
                "(TextCaps-style annotations, five per image) scored with CIDEr / BLEU-4 over a corpus; no "
                "such labelled set ships with this repository"
            ),
        }
    if len(drawn_texts) != len(results):
        raise ValueError(f"drawn_texts has {len(drawn_texts)} entries for {len(results)} results")
    per_image = []
    for result, texts in zip(results, drawn_texts, strict=True):
        if isinstance(texts, str) or not texts:
            raise ValueError("each drawn_texts entry must be a non-empty sequence of strings")
        prediction = str(result["caption"])
        recall = text_recall(prediction, texts)
        per_image.append(
            {
                "image": result.get("image"),
                "prediction": prediction,
                "drawn_texts": list(texts),
                "text_recall": recall["recall"],
                "found": recall["found"],
                "missing": recall["missing"],
            }
        )
    metrics = [
        {
            "id": "text_recall",
            "value": sum(entry["text_recall"] for entry in per_image) / len(per_image),
            "normalisation": "lower-cased, punctuation removed, whitespace collapsed; distinct drawn tokens",
            "relation_to_benchmarks": (
                "fraction of the drawn text's tokens quoted in the caption; not CIDEr or BLEU-4, which "
                "need several reference captions per image and corpus-level statistics"
            ),
            "estimation": f"{len(per_image)} image(s), no dispersion estimate",
        }
    ]
    return {
        **base,
        "metrics": metrics,
        "per_image": per_image,
        "verdict": "sample-sanity",
        "reason": (
            f"{len(per_image)} image(s) whose text you drew yourself; plumbing evidence for the OCR-free "
            "reading path, not a captioning benchmark"
        ),
        "needs": (
            "several human-written reference captions per image from the deployment domain scored with "
            "CIDEr / BLEU-4 over a corpus for any quality claim; TextCaps is not bundled"
        ),
    }


@dataclass
class Pix2StructTextCapsPipeline:
    """``_runner(image, prefix, max_new_tokens)`` returns ``{"caption": str, "new_tokens": int}``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    dtype: str = "float32"
    source: str = "injected"

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Pix2StructTextCapsPipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Pix2StructForConditionalGeneration, Pix2StructProcessor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = Pix2StructProcessor.from_pretrained(location, **common)
        if getattr(processor.image_processor, "is_vqa", False):
            raise RuntimeError("snapshot image processor is the VQA variant (is_vqa=True); refusing")
        model = Pix2StructForConditionalGeneration.from_pretrained(location, dtype=torch.float32, **common)
        model = model.eval().to(resolved_device)

        def runner(image: Image.Image, prefix: str | None, max_new_tokens: int) -> dict[str, Any]:
            if prefix is None:
                inputs = processor(images=image, return_tensors="pt").to(resolved_device)
                prompt_len = 0
            else:
                # Non-VQA processor: the text becomes decoder_input_ids (a caption prefix); nothing is
                # rendered into the image, so no header font is involved.
                inputs = processor(images=image, text=prefix, return_tensors="pt").to(resolved_device)
                prompt_len = int(inputs["decoder_input_ids"].shape[1])
            with torch.inference_mode():
                generated = model.generate(
                    **inputs, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1
                )
            ids = generated[0]
            decoded = processor.decode(ids, skip_special_tokens=True)
            # Unconditional: decoder_start + caption + eos. Conditional: decoder_start, then the prefix
            # ids echoed, then the new ones (observed: budget 1 yields prompt_len + 2 ids).
            new_tokens = int(ids.shape[0]) - prompt_len - 1
            return {"caption": decoded, "new_tokens": max(new_tokens, 0)}

        return cls(runner, resolved_device, "float32", source)

    def caption(
        self,
        image: Image.Image,
        *,
        prefix: str | None = None,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    ) -> dict[str, Any]:
        """Caption one image; ``caption`` is the decoded text, stripped (prefix included when given)."""
        rgb, checked_prefix, checked_tokens = _check_inputs(image, prefix, max_new_tokens)
        raw = self._runner(rgb, checked_prefix, checked_tokens)
        if not isinstance(raw, dict) or "caption" not in raw:
            raise RuntimeError("runner must return a dict with 'caption'")
        new_tokens = int(raw.get("new_tokens", 0))
        return {
            "caption": str(raw["caption"]).strip(),
            "prefix": checked_prefix,
            "image_size": list(rgb.size),
            "new_tokens": new_tokens,
            "truncated": new_tokens >= checked_tokens,
            "generation": {"max_new_tokens": checked_tokens, "do_sample": False, "decoding": DECODING},
            "device": self.device,
            "dtype": self.dtype,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `8`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `61bee0d7e237…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Pix2StructTextCapsPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "pix2struct-textcaps-base",
  "modelId": "google/pix2struct-textcaps-base",
  "revision": "61bee0d7e2378e601b68f853ceee4f7cf99f1b88",
  "files": [
    {
      "path": "README.md",
      "bytes": 7676,
      "sha256": "e5dc83622c2ce6ce1df491db93f6ddc6a7ac27f5988c4a36c432a69e91de0bf0"
    },
    {
      "path": "config.json",
      "bytes": 5009,
      "sha256": "aaebb1cbc1eb952676e1b874b09ea28b2519f654ebf261807b22eaebc43f2023"
    },
    {
      "path": "model.safetensors",
      "bytes": 1129177976,
      "sha256": "771693f7e310a89d77aa7ee4d1f5ea3e0c4adb0b6112cdf25a20da29cc3d0ea7"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 217,
      "sha256": "885e6dd8946290a61b0a808e90cc74dd1a33a011b6482fd213bc7327945305f8"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 2201,
      "sha256": "5c87151ef0f72a99d1f766a4c418bd2a1f90aaa30a8e22fe5eca9641daebb64f"
    },
    {
      "path": "spiece.model",
      "bytes": 851388,
      "sha256": "7fd650335add59bed55a432186ca0437a09e185c2d241faab468a538fe6bcf94"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3265159,
      "sha256": "0af109b23840545ef2c286073f4373959badba1faa73c8557881d5126f6287c9"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 2583,
      "sha256": "5fdb6767a49aca48fdfa43d0279321918185fc4997bdb3ea72bf3a6301a1b43d"
    }
  ],
  "totalBytes": 1133312209
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Pix2StructTextCapsPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Draw the synthetic scenes or optional BYOD

The default sample is **synthetic** and carries its own references: three flat cartoon scenes with text rendered into them using Pillow's bundled font — a red octagonal `STOP` sign on a pole; a storefront whose fascia reads `Blue Fern Bakery` with a small `OPEN` card in the window; a green jersey reading `LIONS` above the number `42` — the same drawings the repository's smoke run used. The strings drawn into each scene are the references for the `text_recall` sanity check later. They are not a labelled dataset, so nothing here is a TextCaps measurement; two of the smoke run's misses are known in advance (the model did not quote `OPEN` or `LIONS`, and called the green jersey blue). The image digests are printed for the record; they depend on the Pillow build's bundled font rendering. BYOD is optional and disabled by default; when enabled, upload one or more images — the drawn text is unknown for them, so the evaluation report will be `not-measurable`.

Two **caller-owned request parameters** are exposed: `max_new_tokens` bounds the caption (`DEFAULT_MAX_NEW_TOKENS = 30` fits any TextCaps-style sentence; `MAX_NEW_TOKENS = 64` is the ceiling), and `caption_prefix` (empty for unconditional captioning; `A picture of` is the upstream README's example — the model continues whatever you start, sometimes mid-word: the smoke run produced `A redoptical sign that says STOP.` from the prefix `A red`, so the default here is unconditional). Nothing is validated in this cell — the next section hands the images to the pipeline's own validation stage, which is the only checker. Look for one dictionary per image naming the sample kind, size, digest and drawn text, plus the budget and prefix.

In [ ]:
import hashlib
import io
import math

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
max_new_tokens = 30  # @param {type:"integer"}
caption_prefix = ''  # @param {type:"string"}


def synthetic_scenes():
    """Three flat cartoon scenes with text rendered in Pillow's bundled font; returns [(name, image, drawn strings)]."""
    sign = Image.new('RGB', (640, 480), (135, 206, 235))  # sky
    d = ImageDraw.Draw(sign)
    d.rectangle([0, 340, 640, 480], fill=(60, 179, 75))  # grass
    d.rectangle([312, 250, 328, 400], fill=(110, 110, 110))  # pole
    cx, cy, r = 320, 160, 110
    octagon = [(cx + r * math.cos(math.radians(22.5 + 45 * k)), cy + r * math.sin(math.radians(22.5 + 45 * k))) for k in range(8)]
    d.polygon(octagon, fill=(200, 30, 30), outline='white', width=6)
    d.text((cx, cy), 'STOP', fill='white', font=ImageFont.load_default(size=64), anchor='mm')
    shop = Image.new('RGB', (640, 480), (230, 230, 220))
    d = ImageDraw.Draw(shop)
    d.rectangle([40, 120, 600, 460], fill=(190, 150, 110))  # facade
    d.rectangle([40, 120, 600, 200], fill=(40, 70, 140))  # fascia
    d.text((320, 160), 'Blue Fern Bakery', fill='white', font=ImageFont.load_default(size=40), anchor='mm')
    d.rectangle([260, 260, 380, 460], fill=(90, 60, 30))  # door
    d.rectangle([80, 240, 220, 400], fill=(200, 230, 250))  # left window
    d.rectangle([420, 240, 560, 400], fill=(200, 230, 250))  # right window
    d.rectangle([120, 300, 180, 340], fill='white')  # card
    d.text((150, 320), 'OPEN', fill=(200, 30, 30), font=ImageFont.load_default(size=22), anchor='mm')
    jersey = Image.new('RGB', (480, 560), (245, 245, 245))
    d = ImageDraw.Draw(jersey)
    d.polygon([(90, 80), (180, 40), (300, 40), (390, 80), (420, 180), (360, 200), (360, 520), (120, 520), (120, 200), (60, 180)], fill=(30, 120, 60))
    d.text((240, 250), 'LIONS', fill='white', font=ImageFont.load_default(size=52), anchor='mm')
    d.text((240, 380), '42', fill=(255, 215, 0), font=ImageFont.load_default(size=120), anchor='mm')
    return [
        ('synthetic_stop_sign_640x480.png', sign, ['STOP']),
        ('synthetic_bakery_640x480.png', shop, ['Blue Fern Bakery', 'OPEN']),
        ('synthetic_jersey_480x560.png', jersey, ['LIONS', '42']),
    ]


if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    samples = []
    for name, data in uploaded.items():
        image = Image.open(io.BytesIO(data))
        image.load()
        samples.append((name, image, None))
    sample_kind = 'BYOD'
else:
    # Deterministic drawings: no randomness, so no seed is needed; the digests depend on the Pillow build's bundled font.
    samples = synthetic_scenes()
    sample_kind = 'synthetic'

names = [name for name, _, _ in samples]
images = [image for _, image, _ in samples]
drawn_texts = [texts for _, _, texts in samples] if sample_kind == 'synthetic' else None
prefix = caption_prefix.strip() or None
digests = {name: hashlib.sha256(np.asarray(image.convert('RGB')).tobytes()).hexdigest() for name, image in zip(names, images)}
for index, (name, image) in enumerate(zip(names, images)):
    print({'sample_kind': sample_kind, 'name': name, 'mode': image.mode, 'size': image.size, 'rgb_sha256': digests[name], 'drawn_text': drawn_texts[index] if drawn_texts else None})
print({'max_new_tokens': max_new_tokens, 'prefix': prefix, 'n_images': len(images)})

## 5. Validate the request → input manifest

`validate_inputs` is the pipeline's public validation stage: it applies exactly the checks `caption` applies — image type and sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` px, an optional prefix that is a non-empty string of at most `MAX_PREFIX_CHARS` characters (whitespace collapsed), and `max_new_tokens` in `[1, MAX_NEW_TOKENS]` — and returns an **input manifest** naming the schema (including the patch-budget preprocessing and the decoding rule), each input's observed mode and size, the checked prefix, the budget and the verdict. The manifest is written to `outputs/pix2struct_textcaps_input_manifest.json`. To show what rejection looks like, the cell also validates a 4-pixel image and records the pipeline's own error message as a finding. Inside the pipeline each image is converted to RGB and scaled to fill at most `MAX_PATCHES` 16×16 patches (aspect ratio preserved); nothing else is dropped or altered. The pipeline cannot tell whether an image contains text worth quoting: that contract is the caller's.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_PATCHES': MAX_PATCHES, 'MAX_PREFIX_CHARS': MAX_PREFIX_CHARS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DECODING': DECODING}})
input_manifest = validate_inputs(images, prefix=prefix, max_new_tokens=max_new_tokens, names=names)
# Demonstrate rejection on a request that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs([Image.new('RGB', (4, 4))])
except ValueError as exc:
    input_manifest['findings'].append({'input': 'tiny-image-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/pix2struct_textcaps_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 6. Caption the images and read the output correctly

`caption` returns, per image, a dict with `caption` (the decoded text, stripped; the prefix is echoed at the start when one was given), the checked `prefix`, `image_size`, `new_tokens` (tokens generated after the prefix, end-of-sequence included), a `truncated` flag that is true when the budget was exhausted, the generation settings and the model identity. **No score exists**: the caption is generated text with no probability and no correctness signal, and a fluent caption is not evidence that it describes the image or that the quoted text is really there. Greedy decoding is deterministic on a fixed device and dtype; CUDA kernel selection can change a token and therefore the rest of the sentence, so GPU and CPU outputs need not match. Each call re-encodes the image at up to 2048 patches, so cost is per image (about 2.8–3.4 s on the reference CPU). As recorded in the model card, the repository's CPU smoke captioned these same drawings `A stop sign is on a pole.`, `A blue sign that says Blue Fern Bakery.` and `A blue shirt with the number 42 on it.` (missing `OPEN` and `LIONS`, and miscolouring the jersey) — and captioned a blank white image `A man is holding a bottle of alcohol and the bottle says 'the man's'.`: the model always produces a caption with quoted text, whether or not there is any.

In [ ]:
import time

results, seconds = [], []
for name, image in zip(names, images):
    t0 = time.time()
    result = pipe.caption(image, prefix=prefix, max_new_tokens=max_new_tokens)
    result['image'] = name
    results.append(result)
    seconds.append(round(time.time() - t0, 2))
print({'device': pipe.device, 'dtype': pipe.dtype, 'seconds_per_image': seconds, 'any_truncated': any(r['truncated'] for r in results)})
for result in results:
    print(f"{result['image']}\n   caption: {result['caption']!r}  ({result['new_tokens']} tokens{', TRUNCATED' if result['truncated'] else ''})")
if any(r['truncated'] for r in results):
    print('A budget was exhausted: that caption is incomplete. Raise max_new_tokens (ceiling MAX_NEW_TOKENS) and rerun.')

## 7. Evaluate → evaluation report

`evaluation_report` is the pipeline's public evaluation stage and always produces a report. No caption quality is reported by default: TextCaps-style metrics (CIDEr, BLEU-4) need several human-written reference captions per image and corpus-level statistics, and this repository ships none (TextCaps is not bundled). The repository's helper `text_recall` — the fraction of the distinct normalised tokens of the strings drawn into an image that appear in its caption, with the found and missing tokens listed — is what the report uses when the drawn text is known: on the synthetic path those strings are text **you drew yourself**, so a high recall proves only that the input contract, patch extraction, forward pass and decoding round-trip and that the model reads rendered text, and the verdict is `sample-sanity`; the two known misses show what a partial recall looks like. On BYOD no drawn text is known, the verdict is `not-measurable`, and the report states what would make the task measurable. The report is written to `outputs/pix2struct_textcaps_evaluation_report.json`.

In [ ]:
report = evaluation_report(results, drawn_texts, sample_kind=sample_kind)
with open('outputs/pix2struct_textcaps_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({k: v for k, v in report.items() if k not in ('metrics', 'per_image')}, indent=2))
for metric in report['metrics']:
    print(f"{metric['id']:12} {metric['value']:.3f}  ({metric['estimation']})")
for entry in report.get('per_image', []):
    print(f"  text_recall {entry['text_recall']:.2f}  {entry['image']} -> {entry['prediction']!r} (found: {entry['found']}, missing: {entry['missing']})")
if report['verdict'] == 'not-measurable':
    print('The text in these images is not known to the notebook, so nothing is scored; read the captions against the images yourself.')

## 8. Export outputs and provenance

Machine-readable JSON preserves every result (image, caption, prefix, `new_tokens`, `truncated`, the budget), the evaluation report, the input manifest, the sample identities, digests and drawn strings, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable model revision, the model licence, and the runtime identity (Python, `torch`, `transformers`, device). The captions are also written as CSV with explicit `image`, `prefix`, `caption`, `new_tokens`, `truncated` columns, and an annotated PNG contact sheet shows each image with its caption printed beneath it for visual inspection (the model returns no location, so nothing is drawn on the images themselves) — a supplement to, not a replacement for, the machine-readable files. No credentials are recorded.

In [ ]:
import csv

thumb_w, thumb_h, panel_h = 320, 280, 44
sheet = Image.new('RGB', (thumb_w * len(images), thumb_h + panel_h), 'white')
draw = ImageDraw.Draw(sheet)
panel_font = ImageFont.load_default(size=13)
for index, (image, result) in enumerate(zip(images, results)):
    thumb = image.convert('RGB').copy()
    thumb.thumbnail((thumb_w, thumb_h))
    sheet.paste(thumb, (index * thumb_w + (thumb_w - thumb.width) // 2, (thumb_h - thumb.height) // 2))
    draw.text((index * thumb_w + 6, thumb_h + 6), result['caption'][:60], fill=(40, 90, 220), font=panel_font)
    if len(result['caption']) > 60:
        draw.text((index * thumb_w + 6, thumb_h + 24), result['caption'][60:120], fill=(40, 90, 220), font=panel_font)
sheet.save('outputs/pix2struct_textcaps_annotated.png')
payload = {
    'predictions': results,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'names': names, 'sizes': [list(image.size) for image in images], 'rgb_sha256': digests, 'drawn_texts': drawn_texts},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/pix2struct_textcaps_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/pix2struct_textcaps_captions.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image', 'prefix', 'caption', 'new_tokens', 'truncated'])
    for result in results:
        writer.writerow([result['image'], result['prefix'] or '', result['caption'], result['new_tokens'], result['truncated']])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The captions are the text the model generates for an image, quoting whatever text it reads from the pixels; nothing in the output scores that text, the model returns no transcript or location, and it captions every image — including a blank one, for which it invented a bottle with writing on it — with equal fluency. On the drawn scenes the `text_recall` values in the evaluation report compare the captions with strings you drew yourself and the verdict is `sample-sanity`, which proves only that the input contract, patch extraction, forward pass and decoding work and that rendered text is read (the repository's smoke run recalled `STOP` fully, `Blue Fern Bakery` but not `OPEN`, and `42` but not `LIONS`); they say nothing about photographs, curved or stylised type, small print, non-Latin scripts, attributes the model hallucinates (it called the green jersey blue), or captions longer than a sentence, and a BYOD result is a per-image observation with the verdict `not-measurable`. **The model captions any image** and stops only at end-of-sequence or the token budget: check `truncated`, and treat a plausible caption with plausible quoted text on an image that has none as the expected failure mode, not an exception. A prefix steers the caption and can be continued mid-word (`A redoptical sign that says STOP.`), so a conditional caption is partly your sentence. The pipeline provides no OCR transcript, no VQA, no localisation, no benchmark evaluation and no training capability.

Successful execution proves that the recorded repository revision's pipeline module, carried in this notebook, can acquire and digest-verify the pinned model, validate the demonstrated request, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** set `caption_prefix` to `A picture of` and compare (the smoke run got `A picture of is shown with a blue sign that says Blue Fern Bakery.` for the storefront and an immediate end-of-sequence for the stop sign); change the drawn strings in `synthetic_scenes` and watch `text_recall` follow; lower `max_new_tokens` to 3 and watch `truncated` turn true on `A stop sign`; enable `USE_BYOD` with photographs of signs you know, then pass the strings you can read yourself as `drawn_texts` to `evaluation_report` to see the verdict switch to `sample-sanity`.

## References

- Repository README: https://github.com/kurtvalcorza/pix2struct-textcaps-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/pix2struct-textcaps-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/pix2struct-textcaps-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/google/pix2struct-textcaps-base
- Upstream code: https://github.com/google-research/pix2struct
- Pix2Struct: Screenshot Parsing as Pretraining for Visual Language Understanding (Lee et al., 2022): https://arxiv.org/abs/2210.03347
- TextCaps: a Dataset for Image Captioning with Reading Comprehension (Sidorov et al., 2020): https://arxiv.org/abs/2003.12462
- CIDEr: Consensus-based Image Description Evaluation (Vedantam, Zitnick, Parikh, 2015): https://arxiv.org/abs/1411.5726